# 86 — Proper Nested-CV Grand Ensemble

nb85 fitted ElasticNetCV on the full OOF stack in-sample → reported RAE 0.218 is misleading.

Here we use **nested scaffold CV**: for each outer fold, fit the meta-learner on the OTHER 4 folds' OOF predictions, then predict on the held-out fold. This gives unbiased OOF predictions for the ensemble.

Expected: real stacked RAE around 0.50–0.52 (slightly better than best individual model).

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
from sklearn.linear_model import ElasticNetCV

EXCLUDE = {"aux_features", "grand_v6", "grand_v6b", "grand_v6c", "grand_v7",
           "grand15","grand18","grand23","grand24","grand25",
           "creative_mega_ensemble", "cliff_role_proba", "chemprop_cliff_mem_proba",
           "chemprop_chembl_nr_multitask", "per_fp_stack"}

oof_files = sorted(DATA_PROCESSED.glob("oof_*.npy"))
oofs, tes, names = [], [], []
for fp in oof_files:
    name = fp.stem.replace("oof_","")
    if name in EXCLUDE: continue
    te_fp = DATA_PROCESSED / f"te_oof_{name}.npy"
    try:
        arr = np.load(fp)
        if arr.ndim > 1: arr = arr[:,0]
        if len(arr) != len(y_tr): continue
        te_v = np.load(te_fp) if te_fp.exists() else None
        if te_v is None or len(te_v) != 513: continue
        if te_v.ndim > 1: te_v = te_v[:,0]
        te_std = float(te_v.std())
        if te_std < 0.4 * float(y_tr.std()): continue
        arr[~np.isfinite(arr)] = y_tr.mean()
        te_v[~np.isfinite(te_v)] = float(np.nanmean(te_v))
        oofs.append(arr); tes.append(te_v); names.append(name)
    except Exception as e:
        print(f"  skip {name}: {e}")

OOF_stack = np.column_stack(oofs)
TE_stack  = np.column_stack(tes)
print(f"Using {len(names)} base models, stack shape {OOF_stack.shape}")
print(f"Models: {names}")


Using 16 base models, stack shape (4139, 16)
Models: ['3d_shape', 'delta_ml', 'free_wilson', 'graph_spreading', 'lgbm_chembl_all_nr_weighted', 'lgbm_chembl_pxr_direct', 'lgbm_cliff_aware_external', 'lgbm_counter_soft', 'lgbm_crc_singleconc_fdr', 'lgbm_full_metrics_baseline', 'lgbm_pubchem_pxr_fixed', 'multi_fp_ensemble', 'multitask_lgbm_heads', 'pseudo_label', 'selectivity_aware', 'smiles_aug']


In [5]:
# Nested-CV: for each outer fold, fit meta on remaining folds
print("=== Nested-CV stacking ===", flush=True)
oof_nested = np.full(len(y_tr), np.nan)

for k, (tr_idx, va_idx) in enumerate(splits):
    # Meta-train: all folds except k
    meta_tr_idx = [i for fold, (ti, _) in enumerate(splits) for i in ti if fold != k]
    meta_va_idx = va_idx  # held-out fold

    meta = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0], cv=5,
                        max_iter=10000, random_state=SEED)
    meta.fit(OOF_stack[meta_tr_idx], y_tr[meta_tr_idx])
    oof_nested[meta_va_idx] = meta.predict(OOF_stack[meta_va_idx])
    fold_rae = rae(y_tr[meta_va_idx], oof_nested[meta_va_idx])
    print(f"  fold {k+1}  val_RAE={fold_rae:.4f}", flush=True)

m_nested = full_metrics(y_tr, oof_nested, cliff_pairs, "nested_cv_ensemble")
m_nested_a = full_metrics(y_tr[active_mask], oof_nested[active_mask],
                           label="nested_cv [active]")
print(f"\nNested-CV OOF RAE: {m_nested['RAE']:.4f}  (compare: grand_v7=0.5189, grand_v6b=0.5281)")


=== Nested-CV stacking ===


  fold 1  val_RAE=0.3793


  fold 2  val_RAE=0.3952


  fold 3  val_RAE=0.4356


  fold 4  val_RAE=0.4283


  fold 5  val_RAE=0.4314


  [nested_cv_ensemble] RAE=0.4108 MAE=0.3737 R²=0.7629 r=0.8735 ρ=0.8293 τ=0.6527
  [nested_cv [active]] RAE=2.6038 MAE=0.5460 R²=-5.5157 r=0.1651 ρ=0.1241 τ=0.0931

Nested-CV OOF RAE: 0.4108  (compare: grand_v7=0.5189, grand_v6b=0.5281)


In [6]:
# Final model: refit meta on all data, predict test
meta_final = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0], cv=5,
                          max_iter=10000, random_state=SEED)
meta_final.fit(OOF_stack, y_tr)
te_preds = np.clip(meta_final.predict(TE_stack), y_tr.min()-0.5, y_tr.max()+0.5)

coef_df = pd.DataFrame({"model": names, "weight": meta_final.coef_}).sort_values("weight", ascending=False)
print("Non-zero weights:")
print(coef_df[coef_df.weight.abs() > 1e-6].to_string(index=False))

np.save(DATA_PROCESSED/"oof_nested_cv_ensemble.npy", oof_nested)
np.save(DATA_PROCESSED/"te_oof_nested_cv_ensemble.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"86_nested_cv_ensemble.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Non-zero weights:
                 model   weight
              delta_ml 0.794368
     multi_fp_ensemble 0.109501
           free_wilson 0.108597
            smiles_aug 0.086384
lgbm_chembl_pxr_direct 0.040972
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\86_nested_cv_ensemble.csv
Test: min=2.61 med=5.05 max=6.07
